In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

d = np.load("../data/processed/windows_fd001.npz")
RUL_CAP = 125.0

def to_tensors(X, y):
    # targets are divided by 125 so the model learns values between 0 and 1
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y / RUL_CAP, dtype=torch.float32)

X_tr, y_tr = to_tensors(d["X_tr"], d["y_tr"])
X_val, y_val = to_tensors(d["X_val"], d["y_val"])
X_test, y_test = to_tensors(d["X_test"], d["y_test"])

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)


class LSTMRegressor(nn.Module):
    def __init__(self, n_features=28, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        out, _ = self.lstm(x)           # out: (batch, 30, hidden)
        return self.head(out[:, -1, :]).squeeze(-1)   # use the last time step


model = LSTMRegressor()
xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape, model(xb).shape)
print("Parameters:", sum(p.numel() for p in model.parameters()))

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/windows_fd001.npz'